# LLM03 Supply Chain - does the mitigation work?

OWASP LLM03's controls are classic software supply-chain hygiene applied to models and their dependencies: **verify artifact integrity (hash)**, **never deserialise untrusted pickles**, **pin and vet dependencies (typosquat detection)**. All deterministic - no model needed.

Course claims are paraphrased from the public OWASP LLM Top-10 taxonomy (owasp.org); no course text is reproduced. Target `qwen2.5:7b` (aligned). Numbers are measured here, not asserted.


In [1]:
import hashlib, pickle, io, pickletools
print("supply-chain controls are deterministic; no model call")


supply-chain controls are deterministic; no model call


## Mitigation 1 - artifact hash verification


In [2]:
good = b"MODEL_WEIGHTS_v1_authentic"
trusted_sha = hashlib.sha256(good).hexdigest()
def verify(blob): return hashlib.sha256(blob).hexdigest() == trusted_sha
tampered = good + b"\x00backdoor"
print("authentic blob verifies:", verify(good))
print("tampered  blob verifies:", verify(tampered), "(must be False)")
hash_ok = verify(good) and not verify(tampered)


authentic blob verifies: True
tampered  blob verifies: False (must be False)


## Mitigation 2 - safe deserialisation (restricted unpickler)


In [3]:
class Evil:
    def __reduce__(self): return (print, ("PWNED: arbitrary code ran on model load",))
payload = pickle.dumps(Evil())

# unsafe load runs the gadget:
ran = {"v": False}
import builtins
_orig = builtins.print
def _spy(*a, **k):
    if a and "PWNED" in str(a[0]): ran["v"] = True
    return _orig(*a, **k)
builtins.print = _spy
try: pickle.loads(payload)
finally: builtins.print = _orig
print("unsafe pickle.loads executed gadget:", ran["v"])

# restricted unpickler blocks it:
class Restricted(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"blocked global {module}.{name}")
blocked = False
try: Restricted(io.BytesIO(payload)).load()
except pickle.UnpicklingError as e: blocked = True; print("restricted unpickler:", e)
pickle_ok = ran["v"] and blocked


PWNED: arbitrary code ran on model load
unsafe pickle.loads executed gadget: True
restricted unpickler: blocked global builtins.print


## Mitigation 3 - typosquat / dependency-confusion detection


In [4]:
KNOWN = {"numpy","pandas","requests","torch","transformers","scikit-learn"}
def edit1(a,b):
    if abs(len(a)-len(b))>1: return False
    import difflib; return difflib.SequenceMatcher(None,a,b).ratio() >= 0.8
def flag(pkg):
    if pkg in KNOWN: return None
    for k in KNOWN:
        if edit1(pkg,k): return f"typosquat of {k}"
    return "unknown package - vet before install"
tscore = 0
for c in ["numpy","numpyy","pandaa","requsts","torchh","internal-llm-utils"]:
    f = flag(c); print(f"  {c:20} -> {f}")
    if c!="numpy" and f: tscore += 1
typo_ok = tscore == 5


  numpy                -> None
  numpyy               -> typosquat of numpy
  pandaa               -> typosquat of pandas
  requsts              -> typosquat of requests
  torchh               -> typosquat of torch
  internal-llm-utils   -> unknown package - vet before install


## Expected vs actual


In [5]:

import pandas as pd
def agree(expected_good, measured_good):
    return "yes" if expected_good==measured_good else "NO - contradicts"

rows = [
 ["artifact hash verification", "blocks tampered artifacts", f"tampered rejected: {hash_ok}", agree(True, hash_ok)],
 ["restricted unpickler", "blocks code-exec on load", f"gadget blocked: {pickle_ok}", agree(True, pickle_ok)],
 ["typosquat detection", "flags near-name packages", f"flagged {tscore}/5 bad names", agree(True, typo_ok)],
]
print(pd.DataFrame(rows, columns=["mitigation","course expects","measured","agree?"]).to_string(index=False))


                mitigation            course expects                measured agree?
artifact hash verification blocks tampered artifacts tampered rejected: True    yes
      restricted unpickler  blocks code-exec on load    gadget blocked: True    yes
       typosquat detection  flags near-name packages   flagged 5/5 bad names    yes


**Reading it:** every LLM03 control is structural and every one holds - the course is correct throughout, because these are decades-old supply-chain controls, not model behaviour.
